In [ ]:
# Importar librerías estándar para análisis numérico y visualización
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# TensorFlow / Keras: API moderna para modelos secuenciales y capas densas
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, GaussianNoise
from tensorflow.keras.datasets import fashion_mnist

In [ ]:
# Fijar semillas para reproducibilidad de resultados
np.random.seed(42)
tf.random.set_seed(42)

### Datos

In [ ]:
# Cargar el dataset Fashion MNIST: 60,000 imágenes de entrenamiento y 10,000 de prueba
# Cada imagen es de 28×28 píxeles en escala de grises
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

In [ ]:
# Visualizar una imagen de ejemplo del conjunto de entrenamiento
plt.imshow(X_train[0], cmap='gray')
plt.title('Ejemplo de Fashion MNIST')
plt.axis('off');

In [ ]:
# Valor máximo de los píxeles en el dataset original (escala de grises: 0-255)
print(f'Valor máximo: {X_train.max()}')
print(f'Valor mínimo: {X_train.min()}')

In [ ]:
# Normalizar los píxeles al rango [0, 1] para que sean compatibles con la activación sigmoid del decoder
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

In [ ]:
# Forma del conjunto de entrenamiento: (muestras, alto, ancho)
print(f'Forma de X_train: {X_train.shape}')
print(f'Forma de X_test:  {X_test.shape}')

### Autoencoder Básico

In [ ]:
# Definir el codificador (encoder): reduce la dimensionalidad de 28×28 (784) a 25 neuronas (bottleneck)
# Cada capa comprime progresivamente la información de la imagen
encoder = Sequential([
    Input(shape=(28, 28)),                         # Entrada: imagen 28×28 en escala de grises
    Flatten(),                                      # Aplanar a un vector de 784 elementos
    Dense(units=400, activation='relu'),           # Primera capa oculta
    Dense(units=200, activation='relu'),           # Segunda capa oculta
    Dense(units=100, activation='relu'),           # Tercera capa oculta
    Dense(units=50, activation='relu'),            # Cuarta capa oculta
    Dense(units=25, activation='relu')             # Bottleneck: representación latente comprimida
])

In [ ]:
# Definir el decodificador (decoder): reconstruye la imagen a partir del bottleneck de 25 neuronas
# La última capa usa sigmoid para obtener píxeles en el rango [0, 1]
decoder = Sequential([
    Input(shape=(25,)),                            # Entrada: vector latente comprimido
    Dense(units=50, activation='relu'),            # Expansión progresiva
    Dense(units=100, activation='relu'),
    Dense(units=200, activation='relu'),
    Dense(units=400, activation='relu'),
    Dense(units=28 * 28, activation='sigmoid'),    # Reconstrucción de los 784 píxeles
    Reshape([28, 28])                              # Dar forma de imagen 28×28
])

In [ ]:
# Construir el autoencoder completo uniendo encoder y decoder
autoencoder = Sequential([encoder, decoder])

In [ ]:
# Compilar el modelo
# binary_crossentropy es adecuada porque los píxeles están en [0, 1] (como probabilidades)
autoencoder.compile(loss='binary_crossentropy', optimizer='adam')

In [ ]:
# Entrenar el autoencoder: la entrada y el target son la misma imagen (reconstrucción)
autoencoder.fit(
    x=X_train,
    y=X_train,
    validation_data=(X_test, X_test),
    batch_size=128,
    epochs=30
)

In [ ]:
# Reconstruir las imágenes del conjunto de prueba usando el autoencoder entrenado
reconstructed_images = autoencoder.predict(X_test, verbose=0)

In [ ]:
# Visualizar comparación: imágenes originales vs reconstruidas
n_images = 5
plt.figure(figsize=(12, 4))
for i in range(n_images):
    # Original
    plt.subplot(2, n_images, i + 1)
    plt.imshow(X_test[i], cmap='gray')
    plt.title('Original')
    plt.axis('off')
    # Reconstruida
    plt.subplot(2, n_images, i + 1 + n_images)
    plt.imshow(reconstructed_images[i], cmap='gray')
    plt.title('Reconstruida')
    plt.axis('off')
plt.tight_layout();

### Denoising Autoencoder

In [ ]:
# Definir el codificador para denoising: misma arquitectura pero con GaussianNoise al inicio
# GaussianNoise(0.2) añade ruido gaussiano con desviación estándar 0.2 SOLO durante entrenamiento
denoising_encoder = Sequential([
    Input(shape=(28, 28)),
    GaussianNoise(0.2),                             # Capa de ruido: solo activa en training=True
    Flatten(),
    Dense(units=400, activation='relu'),
    Dense(units=200, activation='relu'),
    Dense(units=100, activation='relu'),
    Dense(units=50, activation='relu'),
    Dense(units=25, activation='relu')              # Bottleneck
])

In [ ]:
# Decodificador idéntico al anterior; reconstruye a partir del bottleneck
denoising_decoder = Sequential([
    Input(shape=(25,)),
    Dense(units=50, activation='relu'),
    Dense(units=100, activation='relu'),
    Dense(units=200, activation='relu'),
    Dense(units=400, activation='relu'),
    Dense(units=28 * 28, activation='sigmoid'),
    Reshape([28, 28])
])

In [ ]:
# Construir el denoising autoencoder completo
denoising_autoencoder = Sequential([denoising_encoder, denoising_decoder])

In [ ]:
# Compilar con la misma loss y optimizador
denoising_autoencoder.compile(loss='binary_crossentropy', optimizer='adam')

In [ ]:
# Entrenar: la entrada es X_train (con ruido añadido automáticamente por GaussianNoise)
# y el target es X_train limpio, forzando al modelo a aprender a eliminar el ruido
denoising_autoencoder.fit(
    x=X_train,
    y=X_train,
    validation_data=(X_test, X_test),
    batch_size=128,
    epochs=30
)

In [ ]:
# Crear imágenes ruidosas para evaluar el denoising autoencoder
# Usamos una capa GaussianNoise independiente aplicada explícitamente a las imágenes de test
noise_layer = GaussianNoise(0.2)
noisy_test_images = noise_layer(X_test, training=True).numpy()

In [ ]:
# Reconstruir las imágenes limpias a partir de las versiones ruidosas
denoised_images = denoising_autoencoder.predict(noisy_test_images, verbose=0)

In [ ]:
# Visualizar comparación triple: Original | Ruidosa | Denoised
n_images = 5
plt.figure(figsize=(15, 5))
for i in range(n_images):
    # Original
    plt.subplot(3, n_images, i + 1)
    plt.imshow(X_test[i], cmap='gray')
    plt.title('Original')
    plt.axis('off')
    # Ruidosa
    plt.subplot(3, n_images, i + 1 + n_images)
    plt.imshow(noisy_test_images[i], cmap='gray')
    plt.title('Ruidosa')
    plt.axis('off')
    # Denoised
    plt.subplot(3, n_images, i + 1 + 2 * n_images)
    plt.imshow(denoised_images[i], cmap='gray')
    plt.title('Denoised')
    plt.axis('off')
plt.tight_layout();